# Spherical HMI Extrapolation

Minimal spherical HMI notebook using the NF2 Python API. Prepare FITS files locally or customize the optional DRMS download cell.

In [ ]:
from pathlib import Path

RUN_JSOC_DOWNLOAD = False
RUN_TRAINING = False

JSOC_EMAIL = "you@example.org"
HMI_TIME = "2024.05.10_00:00:00_TAI"
CARRINGTON_ROTATION = 2283
FULL_DISK_JSOC_QUERY = f"hmi.B_720s[{HMI_TIME}]"
SYNOPTIC_JSOC_QUERY = f"hmi.synoptic_mr_polfil_720s[{CARRINGTON_ROTATION}]"
FULL_DISK_SEGMENTS = {"Br": "Br", "Bt": "Bt", "Bp": "Bp"}
SYNOPTIC_SEGMENTS = {"Br": "Br", "Bt": "Bt", "Bp": "Bp"}

RUN_DIR = Path("examples/runs/notebooks/spherical_hmi")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
NF2_PATH = RUN_DIR / "extrapolation_result.nf2"
EXPORT_DIR = RUN_DIR / "exports"

FULL_DISK_BR = DATA_DIR / "full_disk.Br.fits"
FULL_DISK_BT = DATA_DIR / "full_disk.Bt.fits"
FULL_DISK_BP = DATA_DIR / "full_disk.Bp.fits"
FULL_DISK_BR_ERR = FULL_DISK_BR
FULL_DISK_BT_ERR = FULL_DISK_BT
FULL_DISK_BP_ERR = FULL_DISK_BP
SYNOPTIC_BR = DATA_DIR / "synoptic.Br.fits"
SYNOPTIC_BT = DATA_DIR / "synoptic.Bt.fits"
SYNOPTIC_BP = DATA_DIR / "synoptic.Bp.fits"

SPHERICAL_SAMPLING = [24, 48, 96]
RADIUS_RANGE_SOLRAD = [1.0, 1.2]

In [ ]:
from pathlib import Path
import urllib.request

from astropy import units as u
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_JSOC_DOWNLOAD:
    import drms
    client = drms.Client(email=JSOC_EMAIL)

    def download_segments(query, segments, prefix):
        export = client.export(query, method="url", protocol="fits", email=JSOC_EMAIL)
        if hasattr(export, "wait"):
            export.wait()
        paths = {}
        for field, segment in segments.items():
            row = export.urls[export.urls["filename"].str.contains(segment, regex=False)].iloc[0]
            target = DATA_DIR / f"{prefix}.{field}.fits"
            urllib.request.urlretrieve(row["url"], target)
            paths[field] = target
        return paths

    fd = download_segments(FULL_DISK_JSOC_QUERY, FULL_DISK_SEGMENTS, "full_disk")
    syn = download_segments(SYNOPTIC_JSOC_QUERY, SYNOPTIC_SEGMENTS, "synoptic")
    FULL_DISK_BR, FULL_DISK_BT, FULL_DISK_BP = fd["Br"], fd["Bt"], fd["Bp"]
    SYNOPTIC_BR, SYNOPTIC_BT, SYNOPTIC_BP = syn["Br"], syn["Bt"], syn["Bp"]

In [ ]:
config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "logging": {"project": "nf2", "name": "Spherical HMI"},
    "data": {
        "geometry": "spherical",
        "boundaries": [
            {"id": "full_disk", "type": "map", "files": {"Br": str(FULL_DISK_BR), "Bt": str(FULL_DISK_BT), "Bp": str(FULL_DISK_BP)}, "errors": {"Br_err": str(FULL_DISK_BR_ERR), "Bt_err": str(FULL_DISK_BT_ERR), "Bp_err": str(FULL_DISK_BP_ERR)}},
            {"id": "synoptic", "type": "map", "files": {"Br": str(SYNOPTIC_BR), "Bt": str(SYNOPTIC_BT), "Bp": str(SYNOPTIC_BP)}},
        ],
        "samplers": [{"id": "random", "type": "random_radial_grouped", "batch_size": 16384, "n_lat_lon_sample": 64, "length": 10000}],
        "validation": [{"id": "sphere", "type": "sphere"}],
        "max_radius": 1.3,
    },
}

if RUN_TRAINING:
    nf2.run(**config)
else:
    config

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.vtk"), fmt="vtk", metrics=["j", "alpha", "free_energy"])

out = nf2.load(NF2_PATH)
field = out.load_spherical(radius_range=np.array(RADIUS_RANGE_SOLRAD) * u.solRad, sampling=SPHERICAL_SAMPLING, metrics=["j", "free_energy"], progress=True)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(field["b"])
j = values(field["metrics"]["j"])
free_energy = values(field["metrics"]["free_energy"])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j)),
    "sigma_J_1e2": float(sigma_J(b, j) * 1e2),
    "total_free_energy_density": float(np.nansum(free_energy)),
}
metrics

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
current_map = np.nansum(vector_norm(j), axis=0)
free_energy_map = np.nansum(free_energy, axis=0)
surface_br = b[0, :, :, 0]

for ax, image, title, cmap in [
    (axs[0], current_map, "Integrated |J|", "magma"),
    (axs[1], free_energy_map, "Free energy", "inferno"),
    (axs[2], surface_br, "Model surface Br", "RdBu_r"),
]:
    kwargs = {"origin": "lower", "aspect": "auto", "cmap": cmap}
    if cmap == "RdBu_r":
        lim = np.nanpercentile(np.abs(image), 99)
        kwargs.update(vmin=-lim, vmax=lim)
    im = ax.imshow(image, **kwargs)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()